This notebook loads all the parquet files into a pandas dataframes.

It also create a pkl file so you can use these dataframes in another notebook without making changes here

In [ ]:
import pandas as pd

# Load Artist related data
artists = pd.read_parquet('./data/mb_artist.parquet')
artist_tags = pd.read_parquet('./data/mb_artist_tag.parquet')
artist_ratings = pd.read_parquet('./data/mb_artist_ratings.parquet')
artist_credit = pd.read_parquet('./data/mb_artist_credit.parquet')

# Load Album related data
albums = pd.read_parquet('./data/mb_album.parquet')
album_tags = pd.read_parquet('./data/mb_album_tag.parquet')
album_ratings = pd.read_parquet('./data/mb_album_ratings.parquet')
album_country = pd.read_parquet('./data/mb_album_country.parquet')
album_label = pd.read_parquet('./data/mb_album_label.parquet')
album_artists = pd.read_parquet('./data/mb_album_artists.parquet')

# Verify the loads
dataframes = {
    "Artists": artists,
    "Artist Tags": artist_tags,
    "Artist Ratings": artist_ratings,
    "Artist Credit": artist_credit,
    "Albums": albums,
    "Album Tags": album_tags,
    "Album Ratings": album_ratings,
    "Album Country": album_country,
    "Album Label": album_label,
    "Album Artists": album_artists
}

for name, df in dataframes.items():
    print(f"✅ {name}: {df.shape[0]:,} rows loaded.")

In [4]:
# Flatten album_tags into {tag_id: tag_count} per album
album_tag_dict = (
    album_tags
    .groupby('album_id')
    .apply(lambda x: dict(zip(x['tag_id'], x['tag_count'])), include_groups=False)
    .reset_index()
    .rename(columns={0: 'album_tags'})
)

# Map artist_tags to albums via artist_credit, then flatten
artist_tag_dict = (
    albums[['id', 'artist_credit']]
    .merge(artist_tags, left_on='artist_credit', right_on='artist_id', how='inner')
    .groupby('id')
    .apply(lambda x: dict(zip(x['tag_id'], x['tag_count'])), include_groups=False)
    .reset_index()
    .rename(columns={'id': 'album_id', 0: 'artist_tags'})
)

# Flatten label_tags into {tag_id: tag_count} per album
label_tag_dict = (
    album_label[['album_id', 'tag_id', 'tag_count']]
    .groupby('album_id')
    .apply(lambda x: dict(zip(x['tag_id'], x['tag_count'])), include_groups=False)
    .reset_index()
    .rename(columns={0: 'label_tags'})
)

print(f'✅ album_tag_dict: {album_tag_dict.shape[0]:,} albums')
print(f'✅ artist_tag_dict: {artist_tag_dict.shape[0]:,} albums')
print(f'✅ label_tag_dict: {label_tag_dict.shape[0]:,} albums')

In [ ]:
# Build final_album_df: one row per album with scalar features
final_album_df = albums.rename(columns={'id': 'album_id', 'name': 'album_name'})
final_album_df = pd.merge(final_album_df, album_ratings, on='album_id', how='left')
final_album_df = pd.merge(final_album_df, album_country, on='album_id', how='left')
final_album_df = pd.merge(final_album_df, album_label[['album_id', 'label_id', 'label_type']], on='album_id', how='left')
final_album_df = pd.merge(final_album_df, album_tag_dict, on='album_id', how='left')
final_album_df = pd.merge(final_album_df, label_tag_dict, on='album_id', how='left')


print(f'✅ final_album_df: {final_album_df.shape[0]:,} rows, {final_album_df.shape[1]} columns')

In [ ]:
# Build final_artist_df: one row per artist with scalar features and tags
final_artist_df = pd.merge(artists, artist_ratings, left_on='id', right_on='artist_id', how='left').drop(columns=['artist_id'])
final_artist_df = pd.merge(
    final_artist_df,
    artist_tags.groupby('artist_id').apply(lambda x: dict(zip(x['tag_id'], x['tag_count'])), include_groups=False).reset_index().rename(columns={0: 'artist_tags'}),
    left_on='id',
    right_on='artist_id',
    how='left'
).drop(columns=['artist_id'])

print(f'✅ final_artist_df: {final_artist_df.shape[0]:,} rows, {final_artist_df.shape[1]} columns')

In [5]:
# Build master_df: join final_album_df with final_artist_df
master_df = pd.merge(
    final_album_df,
    final_artist_df[['id', 'artist_year', 'type', 'area', 'gender', 'rating', 'rating_count', 'artist_tags']].rename(columns={'rating': 'artist_rating', 'rating_count': 'artist_rating_count'}),
    left_on='artist_credit',
    right_on='id',
    how='left'
).drop(columns=['id'])

print(f'\n🚀 master_df: {master_df.shape[0]:,} rows, {master_df.shape[1]} columns')
print(master_df.columns.tolist())


🚀 Master DataFrame ready with 157,945,861 rows.


In [6]:
import os

os.makedirs('./data/pickles', exist_ok=True)

final_album_df.to_pickle('./data/pickles/final_album_df.pkl')
final_artist_df.to_pickle('./data/pickles/final_artist_df.pkl')
master_df.to_pickle('./data/pickles/master_df.pkl')

print("✅ final_album_df, final_artist_df and master_df have been pickled!")

✅ Artist, Album, and Master tables have been pickled!


In [ ]:
for name, df in {
    "artists": artists,
    "artist_tags": artist_tags,
    "artist_ratings": artist_ratings,
    "albums": albums,
    "album_tags": album_tags,
    "album_ratings": album_ratings,
    "album_country": album_country,
    "album_label": album_label,
    "album_artists": album_artists,
    "master_df": master_df
}.items():
    print(f"\n{'='*40}\n{name}\n{'='*40}")
    df.info()